In [1]:
import pandas as pd
import numpy as np
import copy

import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, Adafactor

import os
HF_TOKEN = os.environ.get('HF_TOKEN')

/Users/emilykim/opt/anaconda3/envs/claim-extraction-py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Prompt engineering https://community.openai.com/t/prompt-engineering-for-rag/621495

In [3]:
# Fine-tuning is based on the following:
#    A base model
#    A base tokenizer
#    A set of desired (input, output) pairs
#        Importantly, there is some nuance with how the chat template is applied to the input, output pairs
#        This notebook provides a framework for fine-tuning with a system prompt
#            and a fixed Yes/No question with a fixed Yes/No output.

In [ ]:
cache_dir = "../assets/models"
output_dir = "../assets/models/llama-1b-claim-ft"
model_path = "meta-llama/Llama-3.2-1B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_path, cache_dir=cache_dir, use_safetensors=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    cache_dir=cache_dir,
    use_safetensors=True,
    padding_side="left",
    fix_mistral_regex=True,
)

tokenizer.pad_token = tokenizer.eos_token

In [16]:
train_df = pd.read_csv('../data/ours/train.csv')
train_df = train_df.rename(columns={"text": "sentence"})
train_df['label'] = train_df['label'].map({1: 'Yes', 0: 'No'})
train_df = train_df.filter(items=['sentence', 'label'])
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(train_df['label'].value_counts())
train_df.head()

,sentence,label
0,Hunter Biden had no experience in Ukraine or i...,Yes
1,Donald Trump delivered the largest tax cuts in...,Yes
2,"In Nigeria … in terms of revenue share, 20% go...",Yes
3,Biden has pledged to stop border wall construc...,Yes
4,"After the police shooting of Jacob Blake, Gov....",Yes


In [17]:
class BinaryClassificationTuner:
    def __init__(self, model, tokenizer, train_dataset, messages):
        self.model = model
        self.tokenizer = tokenizer
        self.train_dataset = train_dataset
        self.messages = messages

    def train(self, epochs):
        optimizer = Adafactor(model.parameters(), weight_decay=0.01)
        train_dataset = self._prepare_train_data()
        for train_instance in train_dataset:
            for _ in range(epochs):
                logits = model(train_instance['chat_template_input_ids'], use_cache=False)['logits']
                loss = self._calculate_loss(logits, train_instance['label_input_ids']).mean()
                
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

                print("loss: ", loss.item())

        return 
        
    def _prepare_train_data(self):
        """"
        Returns the train dataset as a list of dictionaries, where each is a record with chat
        """
        train_dataset = self.train_dataset.to_dict(orient='records')
        train_dataset_prepared = []

        for train_instance in train_dataset:
            messages = copy.deepcopy(self.messages)
            for message in messages:
                if message['role'] == "user":
                    message['content'] = message['content'].replace('__SENTENCE__',train_instance['sentence'])
                    break
                
            chat_template_input_ids = tokenizer.apply_chat_template(messages, tokenize=True, continue_final_message=True, add_generation_prompt=False, return_tensors="pt")
            chat_template_input_ids = chat_template_input_ids[0, :-1].reshape(1,-1)
            
            label_input_ids = tokenizer(train_instance['label'], add_special_tokens=False, return_tensors="pt", padding="max_length", max_length=chat_template_input_ids.shape[1])['input_ids']
            label_input_ids = torch.where(label_input_ids != tokenizer.pad_token_id, label_input_ids, -100)

            train_dataset_prepared.append({'chat_template_input_ids': chat_template_input_ids,'label_input_ids': label_input_ids})

        return train_dataset_prepared

    def _calculate_loss(self, logits, labels):
        loss_fn = torch.nn.CrossEntropyLoss(reduction='none')
        cross_entropy_loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
        return cross_entropy_loss

## Demo

In [18]:
messages = [
    {"role": "system", "content": "You are an AI agent used to determine whether or not a sentence is a factual claim. Only respond with Yes or No",},
    {"role": "user", "content": "Is the following sentence a factual claim? __SENTENCE__"},
    {"role": "assistant", "content": ""}
]
bct = BinaryClassificationTuner(model, tokenizer, train_df, messages)
bct.train(epochs=2)

loss:  0.3767450153827667
loss:  0.004740158561617136
loss:  0.0710238441824913
loss:  5.364308890420943e-07


In [19]:
messages = [
    {"role": "system", "content": "You are a yes/no answering bot. Only respond to questions with Yes or No",},
    {"role": "user", "content": "Is the capital of New York state New York City?"},
    {"role": "assistant", "content": ""}
]
answer = "Yes"
chat_template = tokenizer.apply_chat_template(messages, tokenize=False, continue_final_message=True)
chat_template_input_ids = tokenizer.apply_chat_template(messages, tokenize=True, continue_final_message=True, add_generation_prompt=False, return_tensors="pt")
chat_template_input_ids = chat_template_input_ids[0, :-1].reshape(1,-1)

label_tokenized = tokenizer([answer], add_special_tokens=False, return_tensors="pt", padding="max_length", max_length=chat_template_input_ids.shape[1])['input_ids']

# -100 comes from the Llama documentation, recommendation for loss
label_tokenized_fixed = torch.where(label_tokenized != tokenizer.pad_token_id, label_tokenized, -100)

# You can use the following to test what the geneartion would complete
#print(tokenizer.batch_decode(model.generate(chat_template_input_ids, max_new_tokens = 1))[0])

In [20]:
# You can use the following to test what the geneartion would complete
# Test a before resposne
print(tokenizer.batch_decode(model.generate(chat_template_input_ids, max_new_tokens = 1))[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 10 May 2026

You are a yes/no answering bot. Only respond to questions with Yes or No<|eot_id|><|start_header_id|>user<|end_header_id|>

Is the capital of New York state New York City?<|eot_id|><|start_header_id|>assistantYes


In [21]:
optimizer = Adafactor(model.parameters(), weight_decay=0.01)

In [22]:
def calculate_loss(logits, labels):
    loss_fn = torch.nn.CrossEntropyLoss(reduction='none')
    cross_entropy_loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
    return cross_entropy_loss

In [23]:
for _ in range(3):
    logits = model(chat_template_input_ids, use_cache=False)["logits"]
    loss = calculate_loss(logits, label_tokenized_fixed).mean()

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print("loss: ", loss.item())


loss:  9.218596801474632e-07
loss:  0.0
loss:  0.0


In [24]:
print(tokenizer.batch_decode(model.generate(chat_template_input_ids, max_new_tokens = 1))[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 10 May 2026

You are a yes/no answering bot. Only respond to questions with Yes or No<|eot_id|><|start_header_id|>user<|end_header_id|>

Is the capital of New York state New York City?<|eot_id|><|start_header_id|>assistantYes


In [25]:
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

('../assets/models/llama-1b-claim-ft/tokenizer_config.json',
 '../assets/models/llama-1b-claim-ft/special_tokens_map.json',
 '../assets/models/llama-1b-claim-ft/chat_template.jinja',
 '../assets/models/llama-1b-claim-ft/tokenizer.json')